<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/cuba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # 🎧 Consecutive Translation Trainer - ВЕРСІЯ GPU
#
# **Модель:** 'medium'. **ВИКОНУЄТЬСЯ НА CUDA (Швидко).**

# %%
# Install all required dependencies
!pip install -q openai-whisper pydub matplotlib python-docx googletrans==4.0.0rc1

import os
import re
import shutil
import numpy as np
import whisper
from pydub import AudioSegment
import matplotlib.pyplot as plt
from docx import Document
import zipfile
from google.colab import files
from googletrans import Translator

# --- Configuration ---
WHISPER_MODEL = "medium"
MINIMUM_PAUSE_SECONDS = 10
SENTENCE_ENDINGS = ['.', '?', '!']
AUDIO_END_BUFFER_MS = 500
device = "cuda"  # <--- ВИКОРИСТАННЯ CUDA (Швидкий)

# --- Налаштування робочих папок ---
input_folder = "/content/input_audio"
output_folder = "/content/output_material"
temp_folder = "/content/temp_files"

# Повне очищення старих папок перед запуском
if os.path.exists(input_folder): shutil.rmtree(input_folder)
if os.path.exists(output_folder): shutil.rmtree(output_folder)
if os.path.exists(temp_folder): shutil.rmtree(temp_folder)

# Додаткове очищення кореневої папки
for f in os.listdir('/content'):
    if f.endswith(('.png', '.docx', '.mp3', '.zip')):
        try:
            os.remove(os.path.join('/content', f))
        except OSError:
            pass

os.makedirs(input_folder)
os.makedirs(output_folder)
os.makedirs(temp_folder)

# --- 1. Завантаження аудіофайлу ---
print("Завантажте ваш аудіофайл (mp3/wav):")
uploaded = files.upload()

if not uploaded:
    print("❌ Файл не завантажено. Припинення роботи.")
    exit()

audio_original_filename = list(uploaded.keys())[0]
audio_processing_path = os.path.join(input_folder, audio_original_filename)

os.rename(audio_original_filename, audio_processing_path)

audio = AudioSegment.from_file(audio_processing_path)


# --- 2. Перша транскрипція для отримання точного тексту ---
print(f"Завантаження моделі '{WHISPER_MODEL}' на пристрої: {device}")
model_loaded = whisper.load_model(WHISPER_MODEL, device=device)

print("Виконується первинна транскрипція...")
result = model_loaded.transcribe(audio_processing_path)
full_text = result["text"]

# --- 3. Точний поділ тексту на граматичні речення (для валідації) ---
def split_sentences(text):
    sentence_delimiters = re.compile(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=[.?!])\s*')
    sentences = sentence_delimiters.split(text)
    return [s.strip() for s in sentences if s.strip()]

corrected_sentences_list = split_sentences(full_text)
corrected_text = " ".join(corrected_sentences_list)

# --- 4. Вирівнювання аудіо за виправленим текстом ---
print("Повторна обробка для точного вирівнювання з аудіо (Таймінги на рівні слів)...")
result_final = model_loaded.transcribe(audio_processing_path, initial_prompt=corrected_text, word_timestamps=True)
initial_segments = result_final['segments']

# =========================================================
# --- 5. ФІНАЛЬНИЙ ПОДІЛ: НАЙБІЛЬШ НАДІЙНИЙ ПОДІЛ ЗА РЕЧЕННЯМИ (Ваша логіка) ---
# =========================================================

def refine_segments_by_sentences(segments):
    refined_segments = []

    for segment in segments:
        current_segment_words = []

        if not segment.get('words'):
            if segment.get('text', '').strip():
                 refined_segments.append(segment)
            continue

        for word_data in segment['words']:
            word = word_data['word']

            current_segment_words.append(word_data)

            ends_with_punctuation = any(word.strip().endswith(p) for p in SENTENCE_ENDINGS)
            segment_text_so_far = "".join([w['word'] for w in current_segment_words]).strip()

            if ends_with_punctuation or any(segment_text_so_far.endswith(p) for p in SENTENCE_ENDINGS):

                start = current_segment_words[0]['start']
                end = current_segment_words[-1]['end']

                text = "".join([w['word'] for w in current_segment_words]).strip()
                text = re.sub(r'\s+([?.!])', r'\1', text)

                if end > start and text:
                    refined_segments.append({
                        'start': start,
                        'end': end,
                        'text': text,
                    })

                current_segment_words = []

        if current_segment_words:
            start = current_segment_words[0]['start']
            end = current_segment_words[-1]['end']
            text = "".join([w['word'] for w in current_segment_words]).strip()
            text = re.sub(r'\s+([?.!])', r'\1', text)

            if end > start and text:
                 refined_segments.append({
                    'start': start,
                    'end': end,
                    'text': text,
                })

    return refined_segments

semantic_segments = refine_segments_by_sentences(initial_segments)
print(f"✅ Поділ сегментів завершено. Згенеровано {len(semantic_segments)} навчальних сегментів.")


# =========================================================
# --- 6. Обробка аудіо та додавання пауз ---
# =========================================================
output_audio = AudioSegment.silent(duration=0)
timeline = []
pause_duration_ms = MINIMUM_PAUSE_SECONDS * 1000

for i, segment in enumerate(semantic_segments):
    start_time = int(segment['start'] * 1000)

    end_time_raw = int(segment['end'] * 1000)
    end_time_buffered = end_time_raw + AUDIO_END_BUFFER_MS

    end_time = min(end_time_buffered, len(audio))

    text = segment['text'].strip()

    if not text or start_time >= end_time:
        continue

    segment_audio = audio[start_time:end_time]
    output_audio += segment_audio
    output_audio += AudioSegment.silent(duration=pause_duration_ms)

    timeline_entry = f"Chunk {i+1} ({len(text.split())} слів, {segment['end'] - segment['start']:.2f}с): {start_time/1000:.2f}с -> {end_time_raw/1000:.2f}с (+{AUDIO_END_BUFFER_MS}ms buffer) | {text}"
    timeline.append(timeline_entry)

print(f"✅ Обробка аудіо та створення пауз завершено. Тривалість паузи: {MINIMUM_PAUSE_SECONDS}с.")

# --- 7. Генерація файлів (ТОЧНИЙ МАКЕТ DOCX) ---
def create_waveform_image(audio_segment, filename):
    data = np.array(audio_segment.get_array_of_samples())
    channels = audio_segment.channels
    sample_rate = audio_segment.frame_rate
    times = np.linspace(0, len(data) / sample_rate / channels, num=len(data) // channels)

    plt.figure(figsize=(15, 5))
    plt.plot(times, data[::channels])
    plt.title(filename)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.savefig(os.path.join(temp_folder, filename))
    plt.close()

waveform_orig_filename = "waveform_original.png"
waveform_pauses_filename = "waveform_with_pauses.png"
doc_filename = "handout_final.docx"
audio_orig_filename = "original.mp3"
audio_pauses_filename = "with_pauses_final.mp3"
zip_filename = "material_for_students.zip"


create_waveform_image(audio, waveform_orig_filename)
create_waveform_image(output_audio, waveform_pauses_filename)

doc = Document()

# --- Page 1: Повна транскрипція + Діаграма оригінального аудіо ---
doc.add_heading('Сторінка 1: Оригінальна транскрипція', level=2)
doc.add_paragraph(corrected_text)
doc.add_picture(os.path.join(temp_folder, waveform_orig_filename))
doc.add_page_break()

# --- Page 2: Таймінги + Діаграма аудіо з паузами ---
doc.add_heading('Сторінка 2: Точні таймінги та Аудіо з паузами', level=2)
doc.add_paragraph('**Точні таймінги (Consecutive Segments):**')
doc.add_paragraph('\n'.join(timeline))
doc.add_picture(os.path.join(temp_folder, waveform_pauses_filename))
doc.add_page_break()

# --- Page 3: Український переклад ---
doc.add_heading("Сторінка 3: Український переклад", level=2)
translator = Translator()

full_text_from_segments = " ".join([s['text'] for s in semantic_segments])
try:
    ukrainian_translation = translator.translate(full_text_from_segments, dest='uk').text
except Exception as e:
    ukrainian_translation = f"Помилка перекладу: {e}. Перевірте підключення до API."

doc.add_paragraph(ukrainian_translation)

doc.save(os.path.join(output_folder, doc_filename))
output_audio.export(os.path.join(output_folder, audio_pauses_filename), format="mp3")
audio.export(os.path.join(output_folder, audio_orig_filename), format="mp3")

# --- 8. Пакетування та завантаження ---
final_zip_path = os.path.join("/content", zip_filename)

if os.path.exists(os.path.join(output_folder, doc_filename)):
    with zipfile.ZipFile(final_zip_path, "w") as zf:
        zf.write(os.path.join(output_folder, audio_orig_filename), audio_orig_filename)
        zf.write(os.path.join(output_folder, audio_pauses_filename), audio_pauses_filename)
        zf.write(os.path.join(output_folder, doc_filename), doc_filename)

        zf.write(os.path.join(temp_folder, waveform_orig_filename), waveform_orig_filename)
        zf.write(os.path.join(temp_folder, waveform_pauses_filename), waveform_pauses_filename)


    print("✅ Роздаточний матеріал готовий. Завантажте файл **material_for_students.zip**.")
    files.download(final_zip_path)
else:
    print("❌ Обробка не вдалася. ZIP-файл не створено.")